# Seção 4 (aliases, serialização e desserialização) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`02_aliases_e_valores_default`](02_aliases_e_valores_default.ipynb), [`04_desserializando_por_nome_ou_alias`](04_desserializando_por_nome_ou_alias.ipynb) | `populate_by_name` → `validate_by_name` / `validate_by_alias` (2.11+) |
| [`05_aliases_de_serializacao`](05_aliases_de_serializacao.ipynb) | `AliasGenerator` já aceita funções diferentes para entrada e saída |
| [`06_aliases_de_validacao`](06_aliases_de_validacao.ipynb) | `AliasChoices`: vence a ordem **da lista**, não a dos dados; texto em célula de código |
| [`07_serializadores_customizados`](07_serializadores_customizados.ipynb) | `pytz` → `zoneinfo` / `timezone.utc` |
| [`10_projeto_solucao_comentada`](10_projeto_solucao_comentada.ipynb) | `alias=` em vez de `validation_alias` quebra os testes `by_alias`; ida e volta pelo JSON |

## Novas

**1. `exclude` usa o nome do campo — e ignora o alias sem avisar.**

In [1]:
from datetime import date
from pydantic import BaseModel, Field, field_serializer


class Person(BaseModel):
    first_name: str = Field(serialization_alias="firstName")
    tax_id: str = Field(serialization_alias="taxId")


person = Person(first_name="Ada", tax_id="123")
print(f"exclude={{'taxId'}}: {person.model_dump(by_alias=True, exclude={'taxId'})}")
print(f"exclude={{'tax_id'}}: {person.model_dump(by_alias=True, exclude={'tax_id'})}")

exclude={'taxId'}: {'firstName': 'Ada', 'taxId': '123'}
exclude={'tax_id'}: {'firstName': 'Ada'}


O primeiro dump **vaza o CPF** que o código pretendia remover — e não há erro, porque `exclude` aceita nomes desconhecidos. O
mesmo vale para `include`. Sempre nomes de campo (os do Python), mesmo serializando por alias.

**2. `@field_serializer` sem `when_used` muda também o dump em Python.**

In [2]:
class Invoice(BaseModel):
    due: date

    @field_serializer("due")
    def br_format(self, value: date) -> str:
        return value.strftime("%d/%m/%Y")


invoice = Invoice(due=date(2026, 1, 2))
print(f"invoice.model_dump(): {invoice.model_dump()}")
print(f"type(invoice.model_dump()['due']).__name__: {type(invoice.model_dump()['due']).__name__}")

invoice.model_dump(): {'due': '02/01/2026'}
type(invoice.model_dump()['due']).__name__: str


O padrão é `when_used="always"`: quem usava `model_dump()["due"]` como `date` (para calcular atraso, por exemplo) passa a receber
texto. Formato de **apresentação** vai com `when_used="json"` (ou `"json-unless-none"`).

**3. O JSON Schema não conhece o formato customizado — e pode mentir.**

In [3]:
class InvoiceNoHint(BaseModel):
    due: date

    @field_serializer("due")
    def br_format(self, value):                    # sem anotação de retorno
        return value.strftime("%d/%m/%Y")


print(f"com '-> str': {Invoice.model_json_schema(mode='serialization')['properties']['due']}")
print(f"sem anotação: {InvoiceNoHint.model_json_schema(mode='serialization')['properties']['due']}")
print(f"saída real (as duas): {invoice.model_dump_json()}")

com '-> str': {'title': 'Due', 'type': 'string'}
sem anotação: {'format': 'date', 'title': 'Due', 'type': 'string'}
saída real (as duas): {"due":"02/01/2026"}


Com `-> str`, o schema diz só "string" — o cliente não sabe o formato. **Sem** a anotação, é pior: o schema anuncia
`format: date` (ISO, `2026-01-02`) e a API devolve `02/01/2026`; um cliente gerado a partir do schema tenta ler ISO e falha.
Anote sempre o retorno do serializador e, se o formato não for ISO, descreva-o (`WithJsonSchema` / `json_schema_extra`) — ou,
melhor, mantenha ISO na API e formate na interface.

## Veja também

- [`dataclass_factory/04_politicas_e_presets`](../../../dataclass_factory/04_politicas_e_presets.ipynb) (caveats): `alias_generator` só vale na saída com
  `by_alias=True` (`serialize_by_alias`).
- [`dataclass_factory/05_modelos_a_partir_de_especificacao`](../../../dataclass_factory/05_modelos_a_partir_de_especificacao.ipynb) (caveats): dois campos
  disputando o mesmo alias são preenchidos com o mesmo valor.